# Multi-LLM Deliberation Framework for Research Paper Analysis

**Version** : OpenRouter (Multi-Provider Council)

**Author** : Shoeb Shakil Sutar

**Institution** : Symbiosis Institute of Technology, Pune

---

## Overview

This notebook implements the **LLM Council** — a multi-model deliberation framework inspired by Andrej Karpathy's LLM Council project, built independently from scratch and applied specifically to **research paper analysis and academic question answering**.

The system works in three stages:
- **Stage 1** : Four council models independently answer the research query in parallel
- **Stage 2** : Responses are anonymized (blind labeling) to eliminate self-preference bias
- **Stage 3** : A meta-model evaluates all responses and synthesizes a final answer

---

## Council Models (via OpenRouter)

| Model | Provider | Research Strength |
|---|---|---|
| `deepseek/deepseek-v3.2` | DeepSeek | Mathematical and logical reasoning |
| `qwen/qwen3-235b-a22b` | Alibaba | Literature breadth, long context |
| `anthropic/claude-sonnet-4-6` | Anthropic | Academic writing and analysis |
| `google/gemini-3-flash-preview` | Google | Factual grounding, speed |

**Meta-Model (Chairman)** : `openai/gpt-5.4` — responsible for evaluation and synthesis

---

## Requirements

- Python 3.10+
- `openai` library
- OpenRouter API Key (stored as `OpenRouter` in Colab secrets)
- `Text_Model_Judge_Response.txt` uploaded to `/content/`

---

## Step 1 — Install Dependencies

In [ ]:
!pip install openai -q

## Step 2 — Imports

In [ ]:
from openai import OpenAI
import time
import string
import threading

## Step 3 — API Key Setup

> Add your OpenRouter API key to **Colab Secrets** with the name `OpenRouter`.
> Go to: `Tools → Secrets → Add new secret`

In [ ]:
from google.colab import userdata
API_KEY = userdata.get("OpenRouter")

## Step 4 — Configuration

Define council models, meta-model, system prompt, and temperature.

> **Note** : `MODELS` contains the four council members. `META_MODEL` is the Chairman that evaluates and synthesizes.

In [ ]:
# Council members — four research-specialized models
MODELS = {
    "llm_1" : "deepseek/deepseek-v3.2",
    "llm_2" : "qwen/qwen3-235b-a22b",
    "llm_3" : "anthropic/claude-sonnet-4-6",
    "llm_4" : "google/gemini-3-flash-preview"
}

# Chairman — evaluates and synthesizes council responses
META_MODEL = "openai/gpt-5.4"

# Research-focused system prompt applied to all council members
SYSTEM_PROMPT = """You are an expert Academic Research Assistant.
When given a research paper, abstract, or research query, analyze it with
academic rigor — focus on methodology, key findings, limitations,
and real-world implications."""

# Temperature — controls randomness of generation (0.7 balances accuracy and fluency)
TEMPERATURE = 0.7

## Step 5 — Core Functions

### 5.1 `call_llm()` — Single Model API Call

Makes a single API call to any specified model via OpenRouter.
Returns a structured dictionary with model name, content, latency, and status.

In [ ]:
def call_llm (model_name : str, prompt : str) -> dict :
  try :
    client = OpenAI(api_key = API_KEY,
                    base_url = "https://openrouter.ai/api/v1")
    start_time = time.time()

    response = client.chat.completions.create(
        model = model_name,
        messages = [
            {"role" : "system", "content" : SYSTEM_PROMPT},
            {"role" : "user",   "content" : prompt}
        ],
        temperature = TEMPERATURE
    )

    latency = round(time.time() - start_time, 3)

    return {
        "Model"   : model_name,
        "Content" : response.choices[0].message.content,
        "Latency" : latency,
        "Status"  : "Success"
    }

  except Exception as e :
    return {
        "Model"   : model_name,
        "Content" : None,
        "Latency" : None,
        "Status"  : "Failure",
        "Error"   : str(e)
    }

### 5.2 `query_llms()` — Parallel Council Inference

Dispatches the user query to all four council models **simultaneously** using Python threading.
Total latency ≈ slowest model, not the sum of all models.

In [ ]:
def query_llms (prompt : str) -> dict :
  try :
    responses = {}
    threads   = []

    # Worker function — each thread calls one council model
    def worker (key, model_name) :
      responses[key] = call_llm(model_name, prompt)

    # Spawn one thread per council model
    for key, value in MODELS.items() :
      t = threading.Thread(target = worker, args = (key, value))
      threads.append(t)
      t.start()

    # Wait for all threads to complete
    for thread in threads :
      thread.join()

    return responses

  except Exception as e :
    return {
        "Error" : str(e)
    }

### 5.3 `blind_responses()` — Anonymization Layer

Strips all model identity information and replaces each response with a neutral label (Response A, B, C, D).
This eliminates **self-preference bias** — the tendency of a model to favour its own output when it can identify authorship.

In [ ]:
def blind_responses (responses : dict) -> dict :
  try :
    letters        = string.ascii_uppercase
    idx            = 0
    blind_response = {}

    for key, value in responses.items() :
      if value["Status"] == "Success" :
        blind_id               = "Response " + letters[idx]
        blind_response[blind_id] = value["Content"]
        idx = idx + 1

    return blind_response

  except Exception as e :
    return {
        "Error" : str(e)
    }

### 5.4 `final_judge_response()` — Deliberation Layer

Sends the original query and all anonymized responses to the meta-model (Chairman).
The judge prompt instructs the meta-model to evaluate each response on:
- **Correctness** /10
- **Clarity** /10
- **Completeness** /10
- **Academic Depth** /10
- **Total** /40

> Requires `Text_Model_Judge_Response.txt` to be uploaded to `/content/`

In [ ]:
def final_judge_response (prompt : str, blind_response : dict) -> str :
  try :
    # Load judge prompt template from file
    with open("/content/Text_Model_Judge_Response.txt") as file :
      instruction = file.read()

    # Format prompt with user query and anonymized responses
    final_judge_prompt = instruction.format(
        user_prompt     = prompt,
        blind_responses = blind_response
    )

    result = call_llm(META_MODEL, final_judge_prompt)
    return result["Content"]

  except Exception as e :
    return str(e)

### 5.5 `user_input()` — Research Query Interface

Displays a research-focused input menu and collects the user's query.

In [ ]:
def user_input () -> str :
  print("=== Research Paper Assistant (LLM Council) ===")
  print("Options :")
  print("  1. Ask a question about a research topic")
  print("  2. Paste a paper abstract for analysis")
  print("  3. Ask for a literature summary on a topic")
  print()

  prompt = input("Enter your research query : ")
  return prompt

### 5.6 `run_llm_council()` — Master Orchestration Function

Runs the full three-stage pipeline end-to-end:
1. Collect user query
2. Query all council models in parallel
3. Anonymize responses
4. Judge and synthesize final answer

In [ ]:
def run_llm_council () :
  try :
    # Stage 0 : Collect research query
    prompt = user_input()

    # Stage 1 : Parallel inference across all council models
    print("\n--- Querying Council Models ---")
    responses = query_llms(prompt)

    for key, value in responses.items() :
      print(f"  {key} ({value['Model']}) → {value['Status']} | Latency : {value.get('Latency', 'N/A')}s")

    # Stage 2 : Anonymize responses
    blind = blind_responses(responses)

    if len(blind) == 0 :
      print("All Models Failed")
      return

    # Stage 3 : Meta-model evaluation and synthesis
    print("\n--- Council Deliberation Complete ---")
    final_response = final_judge_response(prompt, blind)

    print("\n--- Final Research Analysis (Meta Model) ---")
    print(final_response)

  except Exception as e :
    print("Error : ", str(e))

## Step 6 — Run the LLM Council

> Make sure `Text_Model_Judge_Response.txt` is uploaded to `/content/` before running.

In [ ]:
run_llm_council()

---

## Sample Output

```
=== Research Paper Assistant (LLM Council) ===
Options :
  1. Ask a question about a research topic
  2. Paste a paper abstract for analysis
  3. Ask for a literature summary on a topic

Enter your research query : Abstract of Paper, On Layer Normalization in Transformers

--- Querying Council Models ---
  llm_4 (google/gemini-3-flash-preview) → Success | Latency : 6.81s
  llm_3 (anthropic/claude-sonnet-4-6)   → Success | Latency : 17.018s
  llm_2 (qwen/qwen3-235b-a22b)          → Success | Latency : 23.608s
  llm_1 (deepseek/deepseek-v3.2)        → Success | Latency : 39.471s

--- Council Deliberation Complete ---

--- Final Research Analysis (Meta Model) ---
Response A : Correctness 8/10, Clarity 9/10 ...
Rankings (Best to Worst) : A > D > B > C
Final Synthesized Answer : ...
```

---

## References

1. Vaswani et al. (2017). *Attention is All You Need*. NeurIPS.
2. Karpathy, A. (2025). *LLM Council*. https://github.com/karpathy/llm-council
3. Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench*. arXiv:2306.05685.
4. Dietterich, T. G. (2000). *Ensemble Methods in Machine Learning*. Springer.

---

*Symbiosis Institute of Technology, Pune — MTech AI & ML — 2025-2026*